<a href="https://colab.research.google.com/github/qahtanaa/OnSubGroupFairness/blob/main/ComparisonwOthers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%reset -f

In [2]:
# !pip install aif360

In [1]:
import numpy as np
import pandas as pd
from aif360.metrics import utils
from scipy.sparse import issparse
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random
from sklearn.neighbors import NearestNeighbors
from sympy import Symbol
from sympy.solvers import solve
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from aif360.algorithms.preprocessing import *
from aif360.algorithms.preprocessing.optim_preproc_helpers import distortion_functions, opt_tools
from aif360.algorithms.inprocessing import *
from aif360.algorithms.postprocessing import *
import math
import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier


2026-01-28 09:09:30.010139: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/torch/_functorch/deprecated.py:61: UserWarning: We've integrated functorch into PyTorch. As the final step of the integration, functorch.vmap is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use torch.vmap instead; see the PyTorch 2.0 release notes and/or the torch.func migration guide for more details https://pytorch.org/docs/master/func.migrating.html
  warn_deprecated('vmap', 'torch.vmap')




---

---


---



DATASET

In [2]:
def preprocess_dataset(dataset_path, dataset_type, model):
    if dataset_type == 'German':
        df = pd.read_csv(dataset_path)
        df['age'] = df['age'].apply(lambda age: 1 if age >= 25 else 0)
        df['personal_status'] = df['personal_status'].apply(lambda sex: 1 if sex == 'male' else 0)
        print("German dataset:")
        print(df.head())
        sensitive_attributes = ['personal_status','age']
        label = 'credit'
        privileged = [1, 1]
        unprivileged = [0, 0]
        favorable_label = 1
        unfavorable_label = 2
        groups = [
                  {'name': 'Male Adult', 'attributes': {'personal_status': 1, 'age': 1}},
                  {'name': 'Female Adult', 'attributes': {'personal_status': 0, 'age': 1}},
                  {'name': 'Male Young', 'attributes': {'personal_status': 1, 'age': 0}},
                  {'name': 'Female Young', 'attributes': {'personal_status': 0, 'age': 0}}
              ]
        model = model

    elif dataset_type == 'COMPAS':
        df = pd.read_csv(dataset_path)
        selected_columns = ['sex', 'age_cat', 'race', 'juv_fel_count', 'juv_misd_count',
                            'juv_other_count', 'priors_count', 'c_charge_degree',
                            'c_charge_desc', 'two_year_recid']
        df = df[selected_columns]
        df = df[(df['race'] == 'Caucasian') | (df['race'] == 'African-American')].reset_index(drop=True)
        print("COMPAS dataset:")
        print(df.head())
        sensitive_attributes = ['race','sex']
        label = 'two_year_recid'
        privileged = ['Caucasian', 'Female']
        unprivileged = ['African-American', 'Male']
        favorable_label = 0
        unfavorable_label = 1
        groups = [
                  {'name': 'Caucasian Female', 'attributes': {'race': 1, 'sex': 1}},
                  {'name': 'Black Female', 'attributes': {'race': 0, 'sex': 1}},
                  {'name': 'Causasian Male', 'attributes': {'race': 1, 'sex': 0}},
                  {'name': 'Black Male', 'attributes': {'race': 0, 'sex': 0}}
              ]
        model = model

    elif dataset_type == 'Adult':
        df = pd.read_csv(dataset_path, delimiter=';')
        df['income'] = df['income'].str.strip().replace({'>50K.': '>50K', '<=50K.': '<=50K'})
        df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df.replace('?', np.nan, inplace=True)
        df = df.drop(columns=['fnlwgt', 'education-num'])
        df = df[(df['race'] == 'White') | (df['race'] == 'Black')].reset_index(drop=True)
        print("Adult dataset:")
        print(df.head())
        sensitive_attributes = ['race','sex']
        label = 'income'
        privileged = ['White', 'Male']
        unprivileged = ['Black', 'Female']
        favorable_label = '>50K'
        unfavorable_label = '<=50K'
        groups = [
                  {'name': 'White Male', 'attributes': {'race': 1, 'sex': 1}},
                  {'name': 'Black Male', 'attributes': {'race': 0, 'sex': 1}},
                  {'name': 'White Female', 'attributes': {'race': 1, 'sex': 0}},
                  {'name': 'Black Female', 'attributes': {'race': 0, 'sex': 0}}
              ]
        model = model
    elif dataset_type == 'Hospital':
        df = pd.read_csv(dataset_path, delimiter=',')
        print("Hospital dataset:")
        print(df.head())
        sensitive_attributes = ['gender', 'race']
        label = 'disposition'
        privileged = ['Male', 1]
        unprivileged = ['Female', 0]
        favorable_label = 'Admit'
        unfavorable_label = 'Discharge'
        groups = [
                  {'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}},
                  {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}},
                  {'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}},
                  {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}}
              ]
        model = model

    return df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label, groups, model

In [3]:
##################################################################################
# df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label, groups, model = preprocess_dataset('/content/raw_german_dataset.csv', 'Adult', 'Gradient Boosting')
data_path = "data/"
german_data_path = data_path + 'raw_german_dataset.csv'
compas_data_path = data_path + 'raw_compas_dataset.csv'
adult_data_path = data_path + 'raw_adult_dataset.csv'
hospital_data_path = data_path + 'raw_hospital_dataset.csv'

current_dataset = [hospital_data_path, 'Hospital']
# current_dataset = [german_data_path, 'German']

df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label, \
    groups, model = preprocess_dataset(current_dataset[0], current_dataset[1], 'Neural Network')
# = '/content/raw_german_dataset.csv', 'German'
# = preprocess_dataset('/content/raw_compas_dataset.csv', 'COMPAS')
# = preprocess_dataset('/content/raw_adult_dataset.csv', 'Adult')

#model == 'Logistic Regression':
#model == 'Random Forest':
#model == 'Gradient Boosting':
#model == 'Neural Network':

Hospital dataset:
   esi  age  gender  race maritalstatus  employstatus insurance_status  \
0    4   40    Male     1        Single     Full Time            Other   
1    4   66    Male     0       Married  Not Employed       Commercial   
2    2   66    Male     0       Married  Not Employed       Commercial   
3    2   66    Male     0       Married  Not Employed       Commercial   
4    3   84  Female     0       Widowed       Retired         Medicare   

  disposition arrivalmode      previousdispo  n_edvisits  n_admissions  \
0   Discharge     Walk-in  No previous dispo           0             0   
1   Discharge         Car  No previous dispo           0             0   
2   Discharge     Walk-in          Discharge           1             0   
3   Discharge         Car          Discharge           2             0   
4       Admit     Walk-in          Discharge           1             0   

   n_surgeries  n_meds  n_diff_meds             top_meds  
0            1       0           

In [8]:
df

,esi,age,gender,race,maritalstatus,employstatus,insurance_status,disposition,arrivalmode,previousdispo,n_edvisits,n_admissions,n_surgeries,n_meds,n_diff_meds,top_meds
0,4,40,Male,1,Single,Full Time,Other,Discharge,Walk-in,No previous dispo,0,0,1,0,0,NaN
1,4,66,Male,0,Married,Not Employed,Commercial,Discharge,Car,No previous dispo,0,0,2,0,0,NaN
2,2,66,Male,0,Married,Not Employed,Commercial,Discharge,Walk-in,Discharge,1,0,2,0,0,NaN
3,2,66,Male,0,Married,Not Employed,Commercial,Discharge,Car,Discharge,2,0,2,0,0,NaN
4,3,84,Female,0,Widowed,Retired,Medicare,Admit,Walk-in,Discharge,1,0,5,12,9,meds_cardiovascular
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528815,2,49,Male,1,Single,Disabled,Medicare,Admit,ambulance,Admit,4,2,8,27,12,meds_skinpreps
528816,3,50,Male,1,Single,Disabled,Medicare,Admit,ambulance,Admit,5,3,8,16,10,meds_skinpreps
528817,3,50,Male,1,Single,Disabled,Medicare,Discharge,ambulance,Admit,6,4,8,0,0,NaN
528818,3,50,Male,1,Single,Disabled,Medicare,Admit,ambulance,Discharge,7,4,8,22,11,meds_skinpreps


In [5]:
# df[(df['race'] == 1 and df['disposition'] == 'Discharge')]
race = [0, 1]
gender = ['Male', 'Female']
disposition = ['Admit', 'Discharge']
s = 0
for d in disposition:
    for g in gender:
        for r in race:
            L1 = len(df.loc[(df['race'] == r ) & (df['gender'] == g ) & df['disposition'].eq(d)])
            L2 = len(df.loc[(df['race'] == r ) & (df['gender'] == g )])
            print(" r = ", r, "\tg = ", g, "\t d = ", d, "( ", L1, L2, " )", "( ", round(L1 / L2, 3), " )")
            s += L1
print(s, len(df))


 r =  0 	g =  Male 	 d =  Admit (  25570 107070  ) (  0.239  )
 r =  1 	g =  Male 	 d =  Admit (  50092 131388  ) (  0.381  )
 r =  0 	g =  Female 	 d =  Admit (  31388 137331  ) (  0.229  )
 r =  1 	g =  Female 	 d =  Admit (  56480 153031  ) (  0.369  )
 r =  0 	g =  Male 	 d =  Discharge (  81500 107070  ) (  0.761  )
 r =  1 	g =  Male 	 d =  Discharge (  81296 131388  ) (  0.619  )
 r =  0 	g =  Female 	 d =  Discharge (  105943 137331  ) (  0.771  )
 r =  1 	g =  Female 	 d =  Discharge (  96551 153031  ) (  0.631  )
528820 528820


In [6]:
df.columns

Index(['esi', 'age', 'gender', 'race', 'maritalstatus', 'employstatus',
       'insurance_status', 'disposition', 'arrivalmode', 'previousdispo',
       'n_edvisits', 'n_admissions', 'n_surgeries', 'n_meds', 'n_diff_meds',
       'top_meds'],
      dtype='object')



---

---

---

DATA PREPARATION

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

class DataPreparation():
    """
    ........
    """
    def __init__(self, df, sensitive, label, priv, unpriv, fav, unfav, categorical=[]):
        """
        Construct all necessary attributes for the data preparation.

        df : (pandas DataFrame) containing the data
        sensitive : (list(str)) specifying the column names of all sensitive features
        label : (str) specifying the label column
        priv : (list(dicts)) representation of the privileged groups
        unpriv : (list(dicts)) representation of the unprivileged groups
        fav : (str/int/..) value representing the favorable label
        unfav : (str/int/..) value representing the unfavorable label
        categorical : (list(str)) (optional) specifying column names of categorical features
        """
        self.df = df
        self.sensitive = sensitive
        self.label = label
        self.priv = priv
        self.unpriv = unpriv
        self.fav = fav
        self.unfav = unfav
        self.categorical = categorical

    def detect_missing_values(self):
        """
        Detect rows with missing values and remove them from the DataFrame.
        """
        initial_rows = len(self.df)
        self.df = self.df.dropna()
        removed_rows = initial_rows - len(self.df)

        if removed_rows > 0:
            print(f"Detected {removed_rows} rows with missing values. Removed them.")
        else:
            print("No missing values detected.")  # pass

    def binary_label(self):
        """
        Ensure the decision label and sensitive attributes are encoded as binary, where:
        - Favorable label and privileged groups are encoded as 1.
        - Unfavorable label and unprivileged groups are encoded as 0.
        """
        if len(self.priv) != 2 or len(self.unpriv) != 2:
            raise ValueError("Both 'priv' and 'unpriv' must contain exactly two values.")

        number_label_values = self.df[self.label].nunique()
        if number_label_values == 2:
            print(f"The '{self.label}' column has only two unique values.")
            self.df.loc[:, self.label] = self.df[self.label].replace([self.unfav, self.fav], [0, 1])
        else:
            print(f"The '{self.label}' column does not have exactly two unique values, as it should.")

        # Create mappings for each sensitive attribute
        race_mapping = {self.priv[0]: 1, self.unpriv[0]: 0}
        sex_mapping = {self.priv[1]: 1, self.unpriv[1]: 0}

        # Apply the mappings to the respective columns
        self.df.loc[:, self.sensitive[0]] = self.df[self.sensitive[0]].replace(race_mapping)
        self.df.loc[:, self.sensitive[1]] = self.df[self.sensitive[1]].replace(sex_mapping)

    def find_categorical_attributes(self):
        """
        Identify categorical attributes and encode.
        """
        self.attribute_types = {}

        for column in self.df.columns:
            if column == 'Group':
                continue  # Skip the 'Group' column
            elif column in self.categorical:
                self.attribute_types[column] = 'Categorical'
            elif self.df[column].nunique() == 2:
                self.attribute_types[column] = 'Categorical'
            else:
                num_float = 0
                num_text = 0
                thresh = 0.99
                num_att_in_column = len(self.df[column])

                for value in self.df[column]:
                    try:
                        float(value)
                        num_float += 1
                    except ValueError:
                        num_text += 1

                if num_float / num_att_in_column > thresh:
                    self.attribute_types[column] = 'Numerical'
                else:
                    self.attribute_types[column] = 'Categorical'
        # Boolean
        self.cat_features = []
        for attr in self.attribute_types:
            self.cat_features.append(self.attribute_types[attr] == 'Categorical')

        encoder_dict = dict()
        self.columns_categorical = self.df.columns[self.cat_features]

        for column in self.columns_categorical:
            le = LabelEncoder()
            self.df.loc[:, column] = le.fit_transform(self.df[column].values)
            mapping = dict(zip(le.classes_, range(len(le.classes_))))
            encoder_dict[column] = mapping
        print(encoder_dict, 'encoder dict')
        self.numerical_features = [not feature for feature in self.cat_features]
        self.columns_numerical = self.df.columns[self.numerical_features]

        for column in self.columns_numerical:
            self.df.loc[:, column] = self.df[column].astype(float)

        return self.attribute_types, self.cat_features, self.numerical_features

    def create_group_column(self):
        """
        Create a 'Group' column in the DataFrame based on protected attributes, privileged/unprivileged conditions, and label.
        """
        group_combinations = pd.MultiIndex.from_product([self.df[sensitive].unique() for sensitive in self.sensitive] + [self.df[self.label].unique()], names=self.sensitive + [self.label])
        print(list(enumerate(group_combinations)))
        # Create a mapping between group combinations and their corresponding numbers
        group_mapping = {group: idx for idx, group in enumerate(group_combinations)}
        reverse_group_mapping = {idx: group for group, idx in group_mapping.items()}
        self.df['Group'] = pd.MultiIndex.from_frame(self.df[self.sensitive + [self.label]]).map(group_mapping)

        return reverse_group_mapping

    def train_test_split(self):
        X = self.df.loc[:, self.df.columns != self.label]
        y = self.df[self.label]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=0.3, shuffle=True, stratify=self.df['Group'])  # , random_state=42

        return self.X_train, self.y_train, self.X_test, self.y_test

    def standardization_numerical(self):
        train_dataset_numerical = self.X_train[self.columns_numerical]
        test_dataset_numerical = self.X_test[self.columns_numerical]

        scaler = StandardScaler().fit(train_dataset_numerical)
        train_dataset_scaled_numerical = scaler.transform(train_dataset_numerical)
        test_dataset_scaled_numerical = scaler.transform(test_dataset_numerical)

        self.X_train.loc[:, self.columns_numerical] = train_dataset_scaled_numerical
        self.X_test.loc[:, self.columns_numerical] = test_dataset_scaled_numerical

        self.X_train = pd.concat([self.X_train, self.y_train], axis=1)
        self.X_test = pd.concat([self.X_test, self.y_test], axis=1)
        return self.X_train, self.X_test

    def prepare(self):
        """
        Perform all preprocessing steps.
        """
        self.detect_missing_values()
        self.binary_label()
        self.find_categorical_attributes()
        self.create_group_column()
        self.train_test_split()
        self.standardization_numerical()
        return self




In [8]:
##################################################################################
#use the DataPreparation class to preprocess the dataframe
data_prep = DataPreparation(df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label)
data_prep.prepare()
data_prep.df = data_prep.df.reset_index(drop=True)
X_train, X_test = data_prep.X_train, data_prep.X_test
attribute_types = data_prep.attribute_types
cat_features = data_prep.cat_features
numerical_features = data_prep.numerical_features
reverse_group_mapping = data_prep.create_group_column()
#theoretical_num_groups = len(reverse_group_mapping)
X_train = X_train.reset_index(drop=True)

print(X_train.head())

Detected 373090 rows with missing values. Removed them.
The 'disposition' column has only two unique values.


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_91658/1279187019.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df.loc[:, self.label] = self.df[self.label].replace([self.unfav, self.fav], [0, 1])
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_91658/1279187019.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df.loc[:, self.sensitive[0]] = self.df[self.sensitive[0]].replace(race_mapping)


{'gender': {0: 0, 1: 1}, 'race': {0: 0, 1: 1}, 'maritalstatus': {'Civil Union': 0, 'Divorced': 1, 'Legally Separated': 2, 'Life Partner': 3, 'Married': 4, 'Other': 5, 'Significant Other': 6, 'Single': 7, 'Widowed': 8}, 'employstatus': {'Disabled': 0, 'Full Time': 1, 'Not Employed': 2, 'On Active Military Duty': 3, 'Part Time': 4, 'Retired': 5, 'Self Employed': 6, 'Student - Full Time': 7, 'Student - Part Time': 8}, 'insurance_status': {'Commercial': 0, 'Medicaid': 1, 'Medicare': 2, 'Other': 3, 'Self pay': 4}, 'disposition': {0: 0, 1: 1}, 'arrivalmode': {'Car': 0, 'Other': 1, 'Police': 2, 'Public Transportation': 3, 'Walk-in': 4, 'Wheelchair': 5, 'ambulance': 6}, 'previousdispo': {'AMA': 0, 'Admit': 1, 'Discharge': 2, 'Eloped': 3, 'LWBS after Triage': 4, 'LWBS before Triage': 5, 'No previous dispo': 6, 'Observation': 7, 'Send to L&D': 8, 'Transfer to Another Facility': 9}, 'top_meds': {'meds_analgesicandantihistaminecombination': 0, 'meds_analgesics': 1, 'meds_anesthetics': 2, 'meds_ant

/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_91658/1279187019.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df['Group'] = pd.MultiIndex.from_frame(self.df[self.sensitive + [self.label]]).map(group_mapping)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_91658/1279187019.py:148: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-2.26656335  0.4368853   0.4368853  ...  0.4368853  -0.91483903
  1.78860962]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.X_train.loc[:, self.columns_numerical] = train_dataset_scaled_numerical
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_91658/1279187019

[(0, (0, 0, 1)), (1, (0, 0, 0)), (2, (0, 1, 1)), (3, (0, 1, 0)), (4, (1, 0, 1)), (5, (1, 0, 0)), (6, (1, 1, 1)), (7, (1, 1, 0))]
        esi       age gender  race maritalstatus employstatus  \
0 -2.266563  1.463377      1     1             8            5   
1  0.436885  0.576371      0     1             4            5   
2  0.436885  0.428536      0     0             4            4   
3 -0.914839 -0.359914      1     1             7            1   
4  0.436885  1.561934      0     1             8            5   

  insurance_status arrivalmode previousdispo  n_edvisits  n_admissions  \
0                2           6             1    0.016047      0.256625   
1                2           0             1   -0.469974     -0.460672   
2                2           4             2   -0.307967     -0.460672   
3                0           6             6    0.502069     -0.460672   
4                2           6             2   -0.307967     -0.460672   

   n_surgeries    n_meds  n_diff_me

In [11]:
X_train

,esi,age,gender,race,maritalstatus,employstatus,insurance_status,arrivalmode,previousdispo,n_edvisits,n_admissions,n_surgeries,n_meds,n_diff_meds,top_meds,Group,disposition
0,-2.266563,1.463377,1,1,8,5,2,6,1,0.016047,0.256625,1.803979,3.668734,3.143569,45,6,1
1,0.436885,0.576371,0,1,4,5,2,0,1,-0.469974,-0.460672,-0.267752,-0.538869,-0.291922,5,2,1
2,0.436885,0.428536,0,0,4,4,2,4,2,-0.307967,-0.460672,-0.613041,-0.071358,0.236615,24,0,1
3,-0.914839,-0.359914,1,1,7,1,0,6,6,0.502069,-0.460672,1.458691,-0.538869,-0.556190,24,6,1
4,0.436885,1.561934,0,1,8,5,2,6,2,-0.307967,-0.460672,-0.613041,0.084480,-0.027653,13,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109006,0.436885,0.428536,1,0,8,5,2,0,2,-0.307967,-0.460672,0.422825,0.240317,0.236615,13,4,1
109007,0.436885,-0.409193,0,0,4,2,1,0,2,-0.307967,-0.460672,2.149268,-1.006381,-1.084727,33,0,1
109008,0.436885,1.512656,0,1,4,5,2,0,1,-0.307967,-0.102024,2.839845,-1.006381,-1.084727,25,3,0
109009,-0.914839,0.379258,1,1,4,4,0,4,2,-0.307967,-0.460672,0.077536,-0.694706,-0.556190,35,7,0


In [14]:
# df[(df['race'] == 1 and df['disposition'] == 'Discharge')]
race = [0, 1]
gender = [1, 0]
disposition = [0, 1]
s = 0
for d in disposition:
    for g in gender:
        for r in race:
            L1 = len(X_train.loc[(X_train['race'] == r ) & (X_train['gender'] == g ) & X_train['disposition'].eq(d)])
            L2 = len(X_train.loc[(X_train['race'] == r ) & (X_train['gender'] == g )])
            print(" r = ", r, "\tg = ", g, "\t d = ", d, "( ", L1, L2, " )", "( ", round(L1 / L2, 3), " )")
            s += L1
print(s, len(X_train))

 r =  0 	g =  1 	 d =  0 (  4680 14941  ) (  0.313  )
 r =  1 	g =  1 	 d =  0 (  8774 32165  ) (  0.273  )
 r =  0 	g =  0 	 d =  0 (  8136 22108  ) (  0.368  )
 r =  1 	g =  0 	 d =  0 (  12223 39797  ) (  0.307  )
 r =  0 	g =  1 	 d =  1 (  10261 14941  ) (  0.687  )
 r =  1 	g =  1 	 d =  1 (  23391 32165  ) (  0.727  )
 r =  0 	g =  0 	 d =  1 (  13972 22108  ) (  0.632  )
 r =  1 	g =  0 	 d =  1 (  27574 39797  ) (  0.693  )
109011 109011


In [92]:
#################################################################################
num_privileged_ones = X_train[(X_train[sensitive_attributes[0]] == 1) &
                              (X_train[sensitive_attributes[1]] == 1) &
                              (X_train[label] == 1)].shape[0]

num_privileged_zeros = X_train[(X_train[sensitive_attributes[0]] == 1) &
                               (X_train[sensitive_attributes[1]] == 1) &
                               (X_train[label] == 0)].shape[0]

# Calculating the ratio of the most privileged class
total_ratio = num_privileged_ones / num_privileged_zeros if num_privileged_zeros != 0 else float('inf')  # Avoid division by zero

print(f"Ratio of most privileged class: {total_ratio}")

Ratio of most privileged class: 2.6659448370184635


In [93]:
# Define sensitive attribute and label columns
sensitive_attr_0 = sensitive_attributes[0]
sensitive_attr_1 = sensitive_attributes[1]

# Calculate the number of ones and zeros for each group
num_group_11_ones = X_train[(X_train[sensitive_attr_0] == 1) &
                            (X_train[sensitive_attr_1] == 1) &
                            (X_train[label] == 1)].shape[0]

num_group_11_zeros = X_train[(X_train[sensitive_attr_0] == 1) &
                             (X_train[sensitive_attr_1] == 1) &
                             (X_train[label] == 0)].shape[0]

num_group_10_ones = X_train[(X_train[sensitive_attr_0] == 1) &
                            (X_train[sensitive_attr_1] == 0) &
                            (X_train[label] == 1)].shape[0]

num_group_10_zeros = X_train[(X_train[sensitive_attr_0] == 1) &
                             (X_train[sensitive_attr_1] == 0) &
                             (X_train[label] == 0)].shape[0]

num_group_01_ones = X_train[(X_train[sensitive_attr_0] == 0) &
                            (X_train[sensitive_attr_1] == 1) &
                            (X_train[label] == 1)].shape[0]

num_group_01_zeros = X_train[(X_train[sensitive_attr_0] == 0) &
                             (X_train[sensitive_attr_1] == 1) &
                             (X_train[label] == 0)].shape[0]

num_group_00_ones = X_train[(X_train[sensitive_attr_0] == 0) &
                            (X_train[sensitive_attr_1] == 0) &
                            (X_train[label] == 1)].shape[0]

num_group_00_zeros = X_train[(X_train[sensitive_attr_0] == 0) &
                             (X_train[sensitive_attr_1] == 0) &
                             (X_train[label] == 0)].shape[0]

# Calculate imbalance ratios
imbalance_ratio_11 = num_group_11_ones / num_group_11_zeros if num_group_11_zeros != 0 else float('inf')
imbalance_ratio_10 = num_group_10_ones / num_group_10_zeros if num_group_10_zeros != 0 else float('inf')
imbalance_ratio_01 = num_group_01_ones / num_group_01_zeros if num_group_01_zeros != 0 else float('inf')
imbalance_ratio_00 = num_group_00_ones / num_group_00_zeros if num_group_00_zeros != 0 else float('inf')

# Print imbalance ratios
print(f"Imbalance ratio for group (1, 1): {imbalance_ratio_11}")
print(f"Imbalance ratio for group (1, 0): {imbalance_ratio_10}")
print(f"Imbalance ratio for group (0, 1): {imbalance_ratio_01}")
print(f"Imbalance ratio for group (0, 0): {imbalance_ratio_00}")


Imbalance ratio for group (1, 1): 2.6659448370184635
Imbalance ratio for group (1, 0): 2.1925213675213677
Imbalance ratio for group (0, 1): 2.255910987482615
Imbalance ratio for group (0, 0): 1.7173058013765978


In [94]:
#################################################################################
## Save the 'Group' column from X_train
subgroup_column_train = X_train['Group']
subgroup_column_test = X_test['Group']

# Drop the 'Group' column from X_train
X_train = X_train.drop(columns=['Group'])
X_test = X_test.drop(columns=['Group'])



---



---



---

DBSCAN



In [95]:
# !pip install gower

In [96]:
# find the eps
from gower import gower_matrix
from sklearn.cluster import DBSCAN
import math

# # Filter the dataset for one specific combination of sensitive attributes and labels
# filtered_data = X_train[(X_train[sensitive_attributes[0]] == 1) & (X_train[sensitive_attributes[1]] == 0) & (X_train[label] == 1)]

# # Calculate Gower distance matrix
# distance_matrix = gower_matrix(filtered_data, cat_features=cat_features)

# eps = 0.1
# # DBSCAN clustering
# dbscan = DBSCAN(eps=eps, min_samples=round(math.log(len(filtered_data))), metric='precomputed')  # Set appropriate values for eps and min_samples
# clusters = dbscan.fit_predict(distance_matrix)

# # Assign cluster labels to dataframe
# filtered_data['cluster'] = clusters

# print(round(math.log(len(filtered_data))))
# # Display the number of samples in each cluster
# cluster_counts = filtered_data.groupby(['cluster']).size()
# print("Number of samples in each cluster:")
# print(cluster_counts)

# # Get cluster labels
# labels = dbscan.labels_

# # Get core samples
# core_samples_mask = np.zeros_like(labels, dtype=bool)
# core_samples_mask[dbscan.core_sample_indices_] = True

# # Identify core, border, and noise points
# core_points = filtered_data[core_samples_mask]
# border_points = filtered_data[~core_samples_mask & (labels != -1)]
# noise_points = filtered_data[labels == -1]

# # For example, you can print the number of points in each category
# print("Number of core points:", len(core_points))
# print("Number of border points:", len(border_points))
# print("Number of noise points:", len(noise_points))


In [97]:
from gower import gower_matrix
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

# # Compute distances to nearest neighbors
# k = round(math.log(len(filtered_data)))
# nbrs = NearestNeighbors(n_neighbors=k, metric='precomputed').fit(distance_matrix)
# distances, _ = nbrs.kneighbors(distance_matrix)

# # Compute reachability distances
# reachability_distances = np.mean(distances[:, 1:], axis=1)

# # Sort reachability distances in ascending order
# sorted_distances = np.sort(reachability_distances)

# # Plot reachability distances
# plt.figure(figsize=(6, 4))
# plt.plot(sorted_distances)
# plt.title('Reachability Plot')
# plt.xlabel('Data Points (Sorted)')
# plt.ylabel('Reachability Distance')
# plt.grid(True)
# plt.show()


In [98]:
# Compute distances to K-nearest neighbors
# k = round(math.log(len(filtered_data))) # Choose the value of K
# nbrs = NearestNeighbors(n_neighbors=k, metric='precomputed').fit(distance_matrix)
# distances, _ = nbrs.kneighbors(distance_matrix)

# # Sort distances
# sorted_distances = np.sort(distances[:, -1])

# # Plot K-distance graph
# plt.plot(range(len(filtered_data)), sorted_distances)
# plt.xlabel('Data Points')
# plt.ylabel('Distance to Kth Nearest Neighbor')
# plt.title('K-distance Graph')
# plt.show()


In [99]:
# def custom_smote_dbscan(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio):
#     """
#     X_train is the training dataset preprocessed, subgroup_column_train is a column containing the subgroup of each
#     instance in X_train
#     """
#     cat_attr_ix = [i for i, value in enumerate(cat_features) if value]

#     X2_df = X_train[group_column_train == pu_ix]
#     X2 = X2_df.values
#     X3_df = X_train[group_column_train == nu_ix]
#     X3 = X3_df.values

#     PU = len(X2)
#     NU = len(X3)

#     # Determine the oversampling target based on a given total_ratio
#     if (PU / NU) > total_ratio:
#         oversampling_target = (PU / total_ratio) - NU
#         os_df = X3_df
#         os_ix = nu_ix
#     elif (PU / NU) == total_ratio:
#         print("The ratio of PU to NU is within the acceptable range of total_ratio.")
#         return [], 0, pu_ix
#     else:
#         oversampling_target = (total_ratio * NU) - PU
#         os_df = X2_df
#         os_ix = pu_ix
#     os_df = os_df.reset_index(drop=True)

#     # Calculate Gower distance matrix
#     distance_matrix = gower_matrix(os_df, cat_features=cat_features)

#     # group 0 : eps = 0.23
#     # group 1 : eps = 0.275
#     # group 2 : eps = 0.26
#     # group 3 : eps = 0.26
#     # group 4 : eps = 0.26
#     # group 5 : eps = 0.28
#     # group 6 : eps = 0.28
#     # group 7 : eps = 0.265
#     # DBSCAN parameters per group
#     eps = {0: 0.1, 1: 0.1, 2: 0.1, 3: 0.1, 4: 0.1, 5: 0.1, 6: 0.1, 7: 0.1}
#     #eps = {0: 0.225, 1: 0.26, 2: 0.22, 3: 0.255, 4: 0.255, 5: 0.275, 6: 0.27, 7: 0.25}
#     k = round(math.log(len(os_df)))
#     dbscan = DBSCAN(eps=eps[os_ix], min_samples=k, metric='precomputed')
#     clusters = dbscan.fit_predict(distance_matrix)

#     # Get cluster labels
#     labels = dbscan.labels_

#     # Get core samples
#     core_samples_mask = np.zeros_like(labels, dtype=bool)
#     core_samples_mask[dbscan.core_sample_indices_] = True

#     # Identify core, border, and noise points
#     core_points = os_df[core_samples_mask]
#     border_points = os_df[~core_samples_mask & (labels != -1)]
#     noise_points = os_df[labels == -1]

#     if len(border_points) == 0:
#         border_points = core_points

#     # Initialize synthetic samples list
#     synthetic_samples = []

#     border_indices = border_points.index.tolist()
#     random.shuffle(border_indices)
#     current_index = 0

#     while len(synthetic_samples) < oversampling_target:
#         idx_A = border_indices[current_index % len(border_indices)]
#         #idx_A = core_indices[current_index % len(core_indices)]
#         #print(idx_A, 'indice A')
#         current_index += 1
#         point_A = os_df.loc[idx_A]

#         # Ensure point B is not a noise point
#         distances_to_A = distance_matrix[idx_A]
#         neighbors = np.argsort(distances_to_A)[1:k+1]  # Exclude the point itself
#         valid_neighbors = [idx for idx in neighbors if labels[idx] != -1]  # Exclude noise points

#         if not valid_neighbors:
#             continue  # Skip if no valid neighbors are found

#         idx_B = np.random.choice(valid_neighbors)
#         point_B = os_df.loc[idx_B]

#         synthetic_point = {}
#         for i, col in enumerate(os_df.columns):
#             if cat_features[i]:
#                 neighbor_values = os_df.iloc[valid_neighbors][col].tolist()
#                 synthetic_point[col] = max(set(neighbor_values), key=neighbor_values.count)
#             else:
#                 alpha = np.random.rand()
#                 synthetic_point[col] = point_A[col] + alpha * (point_B[col] - point_A[col])

#         synthetic_samples.append(synthetic_point)

#     return pd.DataFrame(synthetic_samples), len(synthetic_samples), os_ix


In [100]:
def find_optimal_epsilon(filtered_data, cat_features, min_samples, distance_matrix, eps_step=0.001, eps_min=0.01, eps_max=1.1):

    def cluster_count(eps):
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
        labels = dbscan.fit_predict(distance_matrix)
        unique_labels = np.unique(labels)
        n_clusters = len(unique_labels)  # - (1 if -1 in unique_labels else 0)
        return n_clusters, unique_labels

    # Binary search for optimal epsilon
    while eps_max - eps_min > eps_step:
        eps_mid = (eps_min + eps_max) / 2
        n_clusters_mid, labels_mid = cluster_count(eps_mid)
        print(eps_mid, n_clusters_mid, labels_mid, 'mids')

        if n_clusters_mid == 1:
            if -1 in labels_mid:
                eps_min = eps_mid  # Only noise points, increase epsilon
            else:
                eps_max = eps_mid  # Only core points, decrease epsilon
        elif n_clusters_mid > 2:
            eps_min = eps_mid  # More than two clusters, increase epsilon
        else:
            eps_max = eps_mid  # Exactly two clusters, continue search to fine-tune
        print(eps_min, eps_max, 'min, max')
    return eps_max

# Custom SMOTE-DBSCAN function
def custom_smote_dbscan(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio):
    """
    X_train is the training dataset preprocessed, group_column_train is a column containing the group of each
    instance in X_train
    """
    cat_attr_ix = [i for i, value in enumerate(cat_features) if value]

    X2_df = X_train[group_column_train == pu_ix]
    X2 = X2_df.values
    X3_df = X_train[group_column_train == nu_ix]
    X3 = X3_df.values

    PU = len(X2)
    NU = len(X3)

    # Determine the oversampling target based on a given total_ratio
    if (PU / NU) > total_ratio:
        oversampling_target = (PU / total_ratio) - NU
        os_df = X3_df
        os_ix = nu_ix
    elif (PU / NU) == total_ratio:
        print("The ratio of PU to NU is within the acceptable range of total_ratio.")
        return [], 0, pu_ix
    else:
        oversampling_target = (total_ratio * NU) - PU
        os_df = X2_df
        os_ix = pu_ix
    os_df = os_df.reset_index(drop=True)

    # Calculate min_samples
    min_samples = round(math.log(len(os_df)))
    distance_matrix = gower_matrix(os_df, cat_features=cat_features)

    # Find the optimal epsilon for os_df
    optimal_eps = find_optimal_epsilon(os_df, cat_features, min_samples, distance_matrix)

    # DBSCAN clustering with the optimal epsilon
    dbscan = DBSCAN(eps=optimal_eps, min_samples=min_samples, metric='precomputed')
    clusters = dbscan.fit_predict(distance_matrix)

    # Get cluster labels
    labels = dbscan.labels_

    # Get core samples
    core_samples_mask = np.zeros_like(labels, dtype=bool)
    core_samples_mask[dbscan.core_sample_indices_] = True

    # Identify core, border, and noise points
    core_points = os_df[core_samples_mask]
    border_points = os_df[~core_samples_mask & (labels != -1)]
    noise_points = os_df[labels == -1]

    if len(border_points) == 0:
        border_points = core_points

    # Initialize synthetic samples list
    synthetic_samples = []

    border_indices = border_points.index.tolist()
    random.shuffle(border_indices)
    current_index = 0

    while len(synthetic_samples) < oversampling_target:
        idx_A = border_indices[current_index % len(border_indices)]
        current_index += 1
        point_A = os_df.loc[idx_A]

        # Ensure point B is not a noise point
        distances_to_A = distance_matrix[idx_A]
        neighbors = np.argsort(distances_to_A)[1:min_samples+1]  # Exclude the point itself
        valid_neighbors = [idx for idx in neighbors if labels[idx] != -1]  # Exclude noise points

        if not valid_neighbors:
            continue  # Skip if no valid neighbors are found

        idx_B = np.random.choice(valid_neighbors)
        point_B = os_df.loc[idx_B]

        synthetic_point = {}
        for i, col in enumerate(os_df.columns):
            if cat_features[i]:
                neighbor_values = os_df.iloc[valid_neighbors][col].tolist()
                synthetic_point[col] = max(set(neighbor_values), key=neighbor_values.count)
            else:
                alpha = np.random.rand()
                synthetic_point[col] = point_A[col] + alpha * (point_B[col] - point_A[col])

        synthetic_samples.append(synthetic_point)

    return pd.DataFrame(synthetic_samples), len(synthetic_samples), os_ix

# Example usage (assuming you have defined X_train, cat_features, etc.):
# synthetic_samples, num_samples, oversampled_index = custom_smote_dbscan(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio)


In [101]:
def oversample_groups(X_train, cat_features, custom_smote, group_column_train, total_ratio, reverse_group_mapping):
    """
    Function to oversample multiple groups automatically based on group labels.

    Parameters:
    - X_train: Preprocessed training dataset.
    - cat_features: List indicating categorical features.
    - custom_smote: Custom SMOTE function to be used.
    - group_column_train: Column containing the group label for each instance.
    - total_ratio: Desired ratio of positive to negative labels.
    - reverse_group_mapping: Mapping of groups to sensitive attributes and labels.

    Returns:
    - synthetic_samples_matrix: Matrix containing all generated synthetic samples.
    - synthetic_samples_group: Array of group labels for the synthetic samples.
    """

    synthetic_samples = []
    synthetic_samples_group = []

    groups = sorted(group_column_train.unique())
    paired_groups = [(groups[i], groups[i+1]) for i in range(0, len(groups), 2)]

    for group1, group2 in paired_groups:
        ########## Determine pu_ix and nu_ix using reverse_group_mapping ##########
        if reverse_group_mapping[group1][2] == 1:
            pu_ix = group1
            nu_ix = group2
        else:
            pu_ix = group2
            nu_ix = group1
        ##########################################################################

        group_df_pu = X_train[group_column_train == pu_ix]
        group_df_nu = X_train[group_column_train == nu_ix]
        positive_count = group_df_pu[group_df_pu[label] == 1].shape[0]
        negative_count = group_df_nu[group_df_nu[label] == 0].shape[0]

        if positive_count == 0 or negative_count == 0:
            continue

        current_ratio = positive_count / negative_count

        if current_ratio == total_ratio:
            continue  # Skip the most privileged group

        synthetic_points, synthetic_count, os_ix = custom_smote(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio=total_ratio)
        pu_column = np.full((len(synthetic_points), 1), os_ix)
        synthetic_samples.append(synthetic_points)
        synthetic_samples_group.append(pu_column)
        print(f"Oversampling for group pair ({pu_ix}, {nu_ix}): Added {synthetic_count} synthetic samples in {os_ix}.")

    synthetic_samples_matrix = pd.concat(synthetic_samples, ignore_index=True)
    synthetic_samples_group = np.concatenate(synthetic_samples_group)

    return synthetic_samples_matrix, synthetic_samples_group


In [102]:
synthetic_samples_matrix_dbscan, synthetic_samples_group_dbscan = oversample_groups(X_train, cat_features, custom_smote_dbscan, subgroup_column_train, total_ratio, reverse_group_mapping)

# Concatenate the original dataset with the synthetic samples
X_train_resampled_dbscan = pd.concat([X_train, pd.DataFrame(synthetic_samples_matrix_dbscan, columns=X_train.columns)], ignore_index=True)
#subgroup_column_resampled_tax = pd.concat([X_train[group_column_train], pd.Series(synthetic_samples_group_tax.flatten())], ignore_index=True)


0.555 1 [0] mids
0.01 0.555 min, max
0.28250000000000003 1 [0] mids
0.01 0.28250000000000003 min, max
0.14625000000000002 2 [-1  0] mids
0.01 0.14625000000000002 min, max
0.07812500000000001 11 [-1  0  1  2  3  4  5  6  7  8  9] mids
0.07812500000000001 0.14625000000000002 min, max
0.11218750000000002 2 [-1  0] mids
0.07812500000000001 0.11218750000000002 min, max
0.09515625000000003 2 [-1  0] mids
0.07812500000000001 0.09515625000000003 min, max
0.08664062500000003 6 [-1  0  1  2  3  4] mids
0.08664062500000003 0.09515625000000003 min, max
0.09089843750000003 2 [-1  0] mids
0.08664062500000003 0.09089843750000003 min, max
0.08876953125000003 4 [-1  0  1  2] mids
0.08876953125000003 0.09089843750000003 min, max
0.08983398437500004 3 [-1  0  1] mids
0.08983398437500004 0.09089843750000003 min, max
0.09036621093750002 2 [-1  0] mids
0.08983398437500004 0.09036621093750002 min, max
Oversampling for group pair (0, 1): Added 7719 synthetic samples in 0.
0.555 1 [0] mids
0.01 0.555 min, max




---



---



---
CLASSIFICATION


In [103]:
def evaluate_model_performance(X_train, X_test, protected_attributes, label_name, groups, model, weights=None):
    favorable_label = 1.0
    unfavorable_label = 0.0
    X_train[label_name] = X_train[label_name].astype(float)
    X_test[label_name] = X_test[label_name].astype(float)
    # If weights is not provided, create an array of ones with the same length as X_train
    if weights is None:
        weights = np.ones(len(X_train))

    # Create BinaryLabelDatasets
    binary_ds_train = BinaryLabelDataset(df=X_train, label_names=[label_name],
                                         protected_attribute_names=protected_attributes,
                                         favorable_label=favorable_label, unfavorable_label=unfavorable_label)
    binary_ds_test = BinaryLabelDataset(df=X_test, label_names=[label_name],
                                        protected_attribute_names=protected_attributes,
                                        favorable_label=favorable_label, unfavorable_label=unfavorable_label)
    if model == 'Logistic Regression':
        classifier = LogisticRegression(max_iter = 500)
    elif model == 'Random Forest':
        classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    elif model == 'Gradient Boosting':
        classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    elif model == 'Neural Network':
        classifier = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
    else:
        raise ValueError('Choose one classification algorithm between Logistic Regression, Random Forest, Gradient Boosting')
    if model == 'Neural Network':
        classifier.fit(X_train.drop(columns=[label_name]), X_train[label_name])
    else:
        classifier.fit(X_train.drop(columns=[label_name]), X_train[label_name], sample_weight=weights)
    predicted_labels = classifier.predict(X_test.drop(columns=[label_name]))

    X_test_with_predictions = pd.concat([X_test.drop(columns=[label_name]), pd.Series(predicted_labels, name=label_name, index=X_test.index)], axis=1)

    binary_ds_test_pred = BinaryLabelDataset(df=X_test_with_predictions, label_names=[label_name],
                                             protected_attribute_names=protected_attributes,
                                             favorable_label=favorable_label, unfavorable_label=unfavorable_label)

    all_results = {}
    for (group1, group2) in itertools.combinations(groups, 2):
        print(group1, group2, 'gruppi')
        pair_key = f"{group1['name']} vs {group2['name']}"
        all_results[pair_key] = evaluate(
            binary_ds_test, binary_ds_test_pred,
            [group1['attributes']], [group2['attributes']])
        #print([group1['attributes']], [group2['attributes']])


    return all_results, predicted_labels

In [105]:
def evaluate(test_data, pred, priv_group, unpriv_group):
    cm = ClassificationMetric(test_data, pred,
                              unprivileged_groups=unpriv_group,
                              privileged_groups=priv_group)
    dm = BinaryLabelDatasetMetric(pred,
                                  unprivileged_groups=unpriv_group,
                                  privileged_groups=priv_group)

    measure_scores = {
        'Balanced Accuracy': balanced_accuracy_score(test_data.labels, pred.labels),
        'Accuracy': cm.accuracy(),
        'F1 Score': f1_score(test_data.labels.ravel(), pred.labels.ravel()),  # Ensure labels are flat
        'Disparate Impact Ratio': dm.disparate_impact(),
        #'Demographic Parity Difference': cm.statistical_parity_difference(),
        #'Predictive Parity Difference': cm.positive_predictive_value(privileged=True) - cm.positive_predictive_value(privileged=False),
        'Average Odds Difference': cm.average_odds_difference(),
        'Equal Opportunity Difference': cm.equal_opportunity_difference(),
        #'Equalized Odds Difference': cm.average_abs_odds_difference(),
        'Consistency': dm.consistency(),
        #'TPR Difference': cm.true_positive_rate_difference(),
        #'FPR Difference': cm.false_positive_rate_difference(),
        #'TNR Difference': cm.true_negative_rate(privileged=True) - cm.true_negative_rate(privileged=False),
        #'FNR Difference': cm.false_negative_rate_difference(),
    }

    return measure_scores

In [106]:
def compute_metrics(df, actual_labels, predicted_labels):
    """Compute fairness and performance metrics."""
    cm = confusion_matrix(actual_labels, predicted_labels)
    TN, FP, FN, TP = cm.ravel()
    metrics = {
        'Accuracy': accuracy_score(actual_labels, predicted_labels),
        'Precision': precision_score(actual_labels, predicted_labels),
        'Recall': recall_score(actual_labels, predicted_labels),
        'F1 Score': f1_score(actual_labels, predicted_labels),
        'TPR': TP / (TP + FN),
        'FPR': FP / (FP + TN),
        'TNR': TN / (TN + FP),
        'FNR': FN / (FN + TP),
        'TP': TP,
        'FP': FP,
        'TN': TN,
        'FN': FN
    }

    return metrics

In [107]:
##################################################################################
results_orig, pred_labels_orig = evaluate_model_performance(X_train, X_test, sensitive_attributes, label,
                                                            groups, model=model)
#model = 'Random Forest'
#model = 'Gradient Boosting'

# Initialize a list to hold DataFrames
data_frames = []

# Populate the list with DataFrames, each having a unique row index
for key, values in results_orig.items():
    df_part = pd.DataFrame([values], index=[key])
    data_frames.append(df_part)

# Concatenate all DataFrames into a single DataFrame
results_orig_df = pd.concat(data_frames)
results_orig_df.index.name = 'Comparison'

# Print the results DataFrame
print(results_orig_df)

/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/

{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} gruppi
{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


...

{'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


                               Balanced Accuracy  Accuracy  F1 Score  \
Comparison                                                             
White Male vs Others Male                0.79604  0.836961  0.884368   
White Male vs White Female               0.79604  0.836961  0.884368   
White Male vs Others Female              0.79604  0.836961  0.884368   
Others Male vs White Female              0.79604  0.836961  0.884368   
Others Male vs Others Female             0.79604  0.836961  0.884368   
White Female vs Others Female            0.79604  0.836961  0.884368   

                               Disparate Impact Ratio  \
Comparison                                              
White Male vs Others Male                         NaN   
White Male vs White Female                        NaN   
White Male vs Others Female                       NaN   
Others Male vs White Female                       NaN   
Others Male vs Others Female                      NaN   
White Female vs Others F

In [108]:
##################################################################################
results_dbscan, pred_labels_dbscan = evaluate_model_performance(X_train_resampled_dbscan, X_test, sensitive_attributes, label,
                                                            groups, model=model)

# Initialize a list to hold DataFrames
data_frames = []

# Populate the list with DataFrames, each having a unique row index
for key, values in results_dbscan.items():
    df_part = pd.DataFrame([values], index=[key])
    data_frames.append(df_part)

# Concatenate all DataFrames into a single DataFrame
results_dbscan_df = pd.concat(data_frames)
results_dbscan_df.index.name = 'Comparison'

# Print the results DataFrame
print(results_dbscan_df)

/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/

{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} gruppi
{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


...

{'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


                               Balanced Accuracy  Accuracy  F1 Score  \
Comparison                                                             
White Male vs Others Male               0.786969  0.837518  0.886527   
White Male vs White Female              0.786969  0.837518  0.886527   
White Male vs Others Female             0.786969  0.837518  0.886527   
Others Male vs White Female             0.786969  0.837518  0.886527   
Others Male vs Others Female            0.786969  0.837518  0.886527   
White Female vs Others Female           0.786969  0.837518  0.886527   

                               Disparate Impact Ratio  \
Comparison                                              
White Male vs Others Male                         NaN   
White Male vs White Female                        NaN   
White Male vs Others Female                       NaN   
Others Male vs White Female                       NaN   
Others Male vs Others Female                      NaN   
White Female vs Others F



---



---



---

OTHER MITIGATION ALGORITHMS - FAIR-SMOTE, REWEIGHING, GERRYFAIR, REMEDY



---

FAIR-SMOTE

In [109]:
# !git clone https://github.com/joymallyac/Fair-SMOTE.git

fatal: destination path 'Fair-SMOTE' already exists and is not an empty directory.


In [110]:
import sys
# Define the new list of directories
new_path = ['./Fair-SMOTE']  # Adjust the path as needed

# Replace sys.path with the new list
sys.path = new_path
print(sys.path)

from Generate_Samples import generate_samples
from Measure import measure_final_score, calculate_recall, calculate_far, calculate_precision, calculate_accuracy
from SMOTE import smote
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names, but NearestNeighbors was fitted with feature names")


['./Fair-SMOTE']


In [111]:
def oversample_fair_smote(X_train, sensitive_attributes, label):
    # Extracting group counts
    zero_zero_zero = len(X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 0)])
    zero_zero_one = len(X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 1)])
    zero_one_zero = len(X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 0)])
    zero_one_one = len(X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 1)])
    one_zero_zero = len(X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 0)])
    one_zero_one = len(X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 1)])
    one_one_zero = len(X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 0)])
    one_one_one = len(X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 1)])

    # Finding maximum
    maximum = max(zero_zero_zero, zero_zero_one, zero_one_zero, zero_one_one, one_zero_zero, one_zero_one, one_one_zero, one_one_one)
    print(f"Maximum count: {maximum}")

    # Printing which group has maximum count
    if maximum == zero_zero_zero:
        print("zero_zero_zero is maximum")
    elif maximum == zero_zero_one:
        print("zero_zero_one is maximum")
    elif maximum == zero_one_zero:
        print("zero_one_zero is maximum")
    elif maximum == zero_one_one:
        print("zero_one_one is maximum")
    elif maximum == one_zero_zero:
        print("one_zero_zero is maximum")
    elif maximum == one_zero_one:
        print("one_zero_one is maximum")
    elif maximum == one_one_zero:
        print("one_one_zero is maximum")
    elif maximum == one_one_one:
        print("one_one_one is maximum")

    # Calculating number of samples to be increased for each group
    zero_zero_zero_to_be_increased = maximum - zero_zero_zero
    zero_zero_one_to_be_increased = maximum - zero_zero_one
    zero_one_zero_to_be_increased = maximum - zero_one_zero
    zero_one_one_to_be_increased = maximum - zero_one_one
    one_zero_zero_to_be_increased = maximum - one_zero_zero
    one_zero_one_to_be_increased = maximum - one_zero_one
    one_one_zero_to_be_increased = maximum - one_one_zero
    one_one_one_to_be_increased = maximum - one_one_one

    print(f"Counts to be increased for each group:")
    print(f"zero_zero_zero: {zero_zero_zero_to_be_increased}")
    print(f"zero_zero_one: {zero_zero_one_to_be_increased}")
    print(f"zero_one_zero: {zero_one_zero_to_be_increased}")
    print(f"zero_one_one: {zero_one_one_to_be_increased}")
    print(f"one_zero_zero: {one_zero_zero_to_be_increased}")
    print(f"one_zero_one: {one_zero_one_to_be_increased}")
    print(f"one_one_zero: {one_one_zero_to_be_increased}")
    print(f"one_one_one: {one_one_one_to_be_increased}")

    df_zero_zero_zero = X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 0)].copy()
    df_zero_zero_one = X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 1)].copy()
    df_zero_one_zero = X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 0)].copy()
    df_zero_one_one = X_train[(X_train[label] == 0) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 1)].copy()
    df_one_zero_zero = X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 0)].copy()
    df_one_zero_one = X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 0)
                                & (X_train[sensitive_attributes[1]] == 1)].copy()
    df_one_one_zero = X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 0)].copy()
    df_one_one_one = X_train[(X_train[label] == 1) & (X_train[sensitive_attributes[0]] == 1)
                                & (X_train[sensitive_attributes[1]] == 1)].copy()


    df_zero_zero_zero.loc[:, sensitive_attributes[0]] = df_zero_zero_zero[sensitive_attributes[0]].astype(str)
    df_zero_zero_zero.loc[:, sensitive_attributes[1]] = df_zero_zero_zero[sensitive_attributes[1]].astype(str)

    df_zero_zero_one.loc[:, sensitive_attributes[0]] = df_zero_zero_one[sensitive_attributes[0]].astype(str)
    df_zero_zero_one.loc[:, sensitive_attributes[1]] = df_zero_zero_one[sensitive_attributes[1]].astype(str)

    df_zero_one_zero.loc[:, sensitive_attributes[0]] = df_zero_one_zero[sensitive_attributes[0]].astype(str)
    df_zero_one_zero.loc[:, sensitive_attributes[1]] = df_zero_one_zero[sensitive_attributes[1]].astype(str)

    df_zero_one_one.loc[:, sensitive_attributes[0]] = df_zero_one_one[sensitive_attributes[0]].astype(str)
    df_zero_one_one.loc[:, sensitive_attributes[1]] = df_zero_one_one[sensitive_attributes[1]].astype(str)

    df_one_zero_zero.loc[:, sensitive_attributes[0]] = df_one_zero_zero[sensitive_attributes[0]].astype(str)
    df_one_zero_zero.loc[:, sensitive_attributes[1]] = df_one_zero_zero[sensitive_attributes[1]].astype(str)

    df_one_zero_one.loc[:, sensitive_attributes[0]] = df_one_zero_one[sensitive_attributes[0]].astype(str)
    df_one_zero_one.loc[:, sensitive_attributes[1]] = df_one_zero_one[sensitive_attributes[1]].astype(str)

    df_one_one_zero.loc[:, sensitive_attributes[0]] = df_one_one_zero[sensitive_attributes[0]].astype(str)
    df_one_one_zero.loc[:, sensitive_attributes[1]] = df_one_one_zero[sensitive_attributes[1]].astype(str)

    df_one_one_one.loc[:, sensitive_attributes[0]] = df_one_one_one[sensitive_attributes[0]].astype(str)
    df_one_one_one.loc[:, sensitive_attributes[1]] = df_one_one_one[sensitive_attributes[1]].astype(str)

    # Generating samples for each group
    df_zero_zero_zero = generate_samples(zero_zero_zero_to_be_increased, df_zero_zero_zero, 'Germann')
    df_zero_zero_one = generate_samples(zero_zero_one_to_be_increased, df_zero_zero_one, 'Germann')
    df_zero_one_zero = generate_samples(zero_one_zero_to_be_increased, df_zero_one_zero, 'Germann')
    df_zero_one_one = generate_samples(zero_one_one_to_be_increased, df_zero_one_one, 'Germann')
    df_one_zero_zero = generate_samples(one_zero_zero_to_be_increased, df_one_zero_zero, 'Germann')
    df_one_zero_one = generate_samples(one_zero_one_to_be_increased, df_one_zero_one, 'Germann')
    df_one_one_zero = generate_samples(one_one_zero_to_be_increased, df_one_one_zero, 'Germann')
    df_one_one_one = generate_samples(one_one_one_to_be_increased, df_one_one_one, 'Germann')

    # Concatenating dataframes
    X_train_resampled_fair_smote = pd.concat([df_zero_zero_zero, df_zero_zero_one, df_zero_one_zero, df_zero_one_one,
                                              df_one_zero_zero, df_one_zero_one, df_one_one_zero, df_one_one_one])
    X_train_resampled_fair_smote.columns = X_train.columns

    return X_train_resampled_fair_smote


In [112]:
###################################################################################
X_train_resampled_fair_smote = oversample_fair_smote(X_train, sensitive_attributes, label)
# Evaluate model performance
results_fair_smote, pred_labels_fair_smote = evaluate_model_performance(X_train_resampled_fair_smote, X_test, sensitive_attributes, label,
                                                            groups, model=model)
# Initialize a list to hold DataFrames
data_frames = []

# Populate the list with DataFrames, each having a unique row index
for key, values in results_fair_smote.items():
    df_part = pd.DataFrame([values], index=[key])
    data_frames.append(df_part)

# Concatenate all DataFrames into a single DataFrame
results_fair_smote_df = pd.concat(data_frames)
results_fair_smote_df.index.name = 'Comparison'

# Print the results DataFrame
print(results_fair_smote_df)

Maximum count: 27574
one_zero_one is maximum
Counts to be increased for each group:
zero_zero_zero: 19438
zero_zero_one: 15351
zero_one_zero: 22894
zero_one_one: 18800
one_zero_zero: 13602
one_zero_one: 0
one_one_zero: 17313
one_one_one: 4183


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/3963011901.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['0' '0' '0' ... '0' '0' '0']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_zero_zero_zero.loc[:, sensitive_attributes[1]] = df_zero_zero_zero[sensitive_attributes[1]].astype(str)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/3963011901.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1' '1' '1' ... '1' '1' '1']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_zero_zero_one.loc[:, sensitive_attributes[1]] = df_zero_zero_one[sensitive_attributes[1]].astype(str)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/3963011901.py:87: FutureWarning: Setting an item of incompatible dtype is depreca

{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} gruppi
{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


...

{'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


                               Balanced Accuracy  Accuracy  F1 Score  \
Comparison                                                             
White Male vs Others Male               0.763019  0.789336  0.844983   
White Male vs White Female              0.763019  0.789336  0.844983   
White Male vs Others Female             0.763019  0.789336  0.844983   
Others Male vs White Female             0.763019  0.789336  0.844983   
Others Male vs Others Female            0.763019  0.789336  0.844983   
White Female vs Others Female           0.763019  0.789336  0.844983   

                               Disparate Impact Ratio  \
Comparison                                              
White Male vs Others Male                         NaN   
White Male vs White Female                        NaN   
White Male vs Others Female                       NaN   
Others Male vs White Female                       NaN   
Others Male vs Others Female                      NaN   
White Female vs Others F



---

REWEIGHTING

In [113]:
# !git clone https://github.com/IBM/AIF360.git

fatal: destination path 'AIF360' already exists and is not an empty directory.


In [114]:
new_path = ['./']
sys.path = new_path
from reweighing_cust import Reweighing

In [115]:
class CustomDataset:
    def __init__(self, data, sensitive_attributes, label):
        self.data = data
        self.protected_attribute_names = sensitive_attributes
        self.protected_attributes = np.column_stack([data[sensitive_attributes[0]], data[sensitive_attributes[1]]])
        self.labels = data[label].astype(float).values.reshape(-1, 1)
        self.favorable_label = 1.0
        self.unfavorable_label = 0.0
        self.instance_weights = np.ones(len(data))
        self.privileged_protected_attributes = [np.array([1.]), np.array([1.])]
        self.unprivileged_protected_attributes = [np.array([0.]), np.array([0.])]
        self.feature_names = data.columns[data.columns != label].tolist()

custom_data = CustomDataset(X_train, sensitive_attributes, label)

In [116]:
privileged_groups = [{sensitive_attributes[0]: 1, sensitive_attributes[1]: 1}]
unprivileged_groups = [{sensitive_attributes[0]: 0, sensitive_attributes[1]: 0}]
RW = Reweighing(unprivileged_groups=unprivileged_groups,
               privileged_groups=privileged_groups)

RW.fit(custom_data)
dataset_transf_train = RW.transform(custom_data)


In [117]:
results_reweighing, pred_labels_reweighing = evaluate_model_performance(X_train, X_test, sensitive_attributes, label,
                                                                        groups, model = model,
                                                                        weights=custom_data.instance_weights)
# Initialize a list to hold DataFrames
data_frames = []

# Populate the list with DataFrames, each having a unique row index
for key, values in results_reweighing.items():
    df_part = pd.DataFrame([values], index=[key])
    data_frames.append(df_part)

# Concatenate all DataFrames into a single DataFrame
results_reweighing_df = pd.concat(data_frames)
results_reweighing_df.index.name = 'Comparison'

# Print the results DataFrame
print(results_reweighing_df)

/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


...

{'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


                               Balanced Accuracy  Accuracy  F1 Score  \
Comparison                                                             
White Male vs Others Male                0.79604  0.836961  0.884368   
White Male vs White Female               0.79604  0.836961  0.884368   
White Male vs Others Female              0.79604  0.836961  0.884368   
Others Male vs White Female              0.79604  0.836961  0.884368   
Others Male vs Others Female             0.79604  0.836961  0.884368   
White Female vs Others Female            0.79604  0.836961  0.884368   

                               Disparate Impact Ratio  \
Comparison                                              
White Male vs Others Male                         NaN   
White Male vs White Female                        NaN   
White Male vs Others Female                       NaN   
Others Male vs White Female                       NaN   
Others Male vs Others Female                      NaN   
White Female vs Others F



---

REMEDY

In [118]:
import sys
sys.path.append('./')  # Add the directory containing remedy_cust.py to the Python path
from remedy_cust import *

In [119]:
columns_all = X_train.drop(columns=[label])
label_y = label
columns_protected = sensitive_attributes

In [120]:
#names = ["Output of get_temp function, all of the attributes for given group"]
#temp2 = ["Output of get_temp function, sum count by group"]
temp2, names = get_temp(X_train, columns_protected, label_y)
print(temp2, names)
#temp2 counts the number of instances for each combination of sensitive attr and label (like my group count)
#names is the list of sensit attr
unfair_group, unfair_names, skew_candidates, unfair_dict = get_unfair_group(columns_protected, [])
#all empty beside skew candidates that is the list of sensitive attr
print(unfair_group, unfair_names, skew_candidates, unfair_dict)
#all_names is a dict that has 0:[], 1:sens attr1, 2:sens attr2, 3:sens attr1, sens attr2. So for each key (number) it
#associates all possible combinations of sens attr
all_names = candidate_groups(skew_candidates, unfair_dict, columns_protected, unfair_names)
#name values is a dict with sens attrib: possible values
names_values = name_val_dict(X_train, names)

all_names_lst = list(all_names.keys())[1:] # CHANGED HERE
all_names_lst.reverse()
#all_names_lst is a list of the keys of the dict all_names, so numbers [3,2,1]
all_names_lst

['gender', 'race', 'disposition']
   gender  race  disposition    cnt
0       0     0          0.0   8136
1       0     0          1.0  13972
2       0     1          0.0  12223
3       0     1          1.0  27574
4       1     0          0.0   4680
5       1     0          1.0  10261
6       1     1          0.0   8774
7       1     1          1.0  23391 ['gender', 'race']
[] [] ['gender', 'race'] {}


/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['cnt'] = 0


[3, 2, 1]

In [121]:
#get all of the candidate groups possible with the combos and names
filter_count = 30
#copy of the dataset
new_train_data = copy.deepcopy(X_train)
print(new_train_data.shape[0])

#iterate over all the names to get the temp2 df for each name
for a in all_names_lst:
  print("?????/////")
  print(a)
  #temp2 counts the number of instances for each combination of sensitive attr and label (like my group count)
  #names is the list of sensit attr
  temp2, names = get_temp(new_train_data, all_names[a], label_y)
  print(temp2, '\n', names, 'temp2,names \n')
  #temp is a df with the entire columns, where the columns are in the first iteration all sens attr and the label
  # in the second iteration are one sens attrib and the label
  # temp_g counts the instances considering only the sens attrib (first iteration both sens attr, then only one at the time)
  temp, temp_g = get_temp_g(new_train_data, names, label_y)
  print(temp,'\n', temp_g, 'temp,temp_g \n')
  temp_g = temp_g[temp_g['cnt'] > filter_count]
  #lst_of_counts is a list of df, the first df have first sens attr and the credit + count, the second is second sens attr
  #and the credit + count
  lst_of_counts = compute_lst_of_counts(temp, names, label_y)
  print(lst_of_counts, 'listof counts \n')

  need_pos, need_neg = compute_problematic_opt(temp2, temp_g, names, label_y, lst_of_counts)
  print("The sets of need pos and neg are")
  print(need_pos)
  print(need_neg)
  new_train_data['skewed'] = 0
  new_train_data["diff"] = 0
  print("started duplication")
  new_train_data = naive_duplicate(new_train_data, temp2, names, need_pos, need_neg, label_y)
  print(new_train_data.shape[0])
  print("label y ", new_train_data[label_y].value_counts())
#new_train_x = pd.DataFrame(new_train_data, columns = columns_all)
new_train_label = pd.DataFrame(new_train_data, columns = [label_y])
new_train_label = new_train_label[label_y]
new_train_label = new_train_label.astype('int')

109011
?????/////
3
['gender', 'race', 'disposition']
   gender  race  disposition    cnt
0       0     0          0.0   8136
1       0     0          1.0  13972
2       0     1          0.0  12223
3       0     1          1.0  27574
4       1     0          0.0   4680
5       1     0          1.0  10261
6       1     1          0.0   8774
7       1     1          1.0  23391 
 ['gender', 'race'] temp2,names 

       gender  race  disposition  cnt
0           1     1          1.0    0
1           0     0          0.0    0
2           1     1          1.0    0
3           0     0          0.0    0
4           0     0          1.0    0
...       ...   ...          ...  ...
109006      0     1          1.0    0
109007      1     1          0.0    0
109008      1     1          1.0    0
109009      1     0          1.0    0
109010      0     1          1.0    0

[109011 rows x 4 columns] 
    gender  race    cnt
0       0     0  22108
1       0     1  39797
2       1     0  14941
3       1 

/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['cnt'] = 0
/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['cnt'] = 0


114926
label y  disposition
1.0    79437
0.0    35489
Name: count, dtype: int64
?????/////
2
['race', 'disposition']


/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['cnt'] = 0


   race  disposition    cnt
0     0          0.0  12816
1     0          1.0  28472
2     1          0.0  22673
3     1          1.0  50965 
 ['race'] temp2,names 

        race  disposition  cnt
0          1          1.0    0
1          0          0.0    0
2          1          1.0    0
3          0          0.0    0
4          0          1.0    0
...      ...          ...  ...
114921     1          0.0    0
114922     1          0.0    0
114923     1          0.0    0
114924     1          0.0    0
114925     1          0.0    0

[114926 rows x 3 columns] 
    race    cnt
0     0  41288
1     1  73638 temp,temp_g 

[disposition
0.0    35489
1.0    79437
Name: cnt, dtype: int64] listof counts 

The sets of need pos and neg are
[]
[]
started duplication
114926
label y  disposition
1.0    79437
0.0    35489
Name: count, dtype: int64
?????/////
1
['gender', 'disposition']
   gender  disposition    cnt
0       0          0.0  20359
1       0          1.0  45785
2       1          0.0  151

/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['cnt'] = 0
/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp['cnt'] = 0
/Users/hakim/git/student_work/OSFFR/./remedy_cust.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pand

In [122]:
X_train_resampled_remedy = new_train_data.drop(columns= ['skewed', 'diff'])
# Evaluate model performance
results_remedy, pred_labels_remedy = evaluate_model_performance(X_train_resampled_remedy, X_test, sensitive_attributes,
                                                                label, groups, model= model)

/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Male', 'attributes': {'race': 1, 'gender': 'Male'}} {'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


...

/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


In [123]:
# Initialize a list to hold DataFrames
data_frames = []

# Populate the list with DataFrames, each having a unique row index
for key, values in results_remedy.items():
    df_part = pd.DataFrame([values], index=[key])
    data_frames.append(df_part)

# Concatenate all DataFrames into a single DataFrame
results_remedy_df = pd.concat(data_frames)
results_remedy_df.index.name = 'Comparison'

# Print the results DataFrame
print(results_remedy_df)

                               Balanced Accuracy  Accuracy  F1 Score  \
Comparison                                                             
White Male vs Others Male               0.795785  0.836897  0.884356   
White Male vs White Female              0.795785  0.836897  0.884356   
White Male vs Others Female             0.795785  0.836897  0.884356   
Others Male vs White Female             0.795785  0.836897  0.884356   
Others Male vs Others Female            0.795785  0.836897  0.884356   
White Female vs Others Female           0.795785  0.836897  0.884356   

                               Disparate Impact Ratio  \
Comparison                                              
White Male vs Others Male                         NaN   
White Male vs White Female                        NaN   
White Male vs Others Female                       NaN   
Others Male vs White Female                       NaN   
Others Male vs Others Female                      NaN   
White Female vs Others F



---



---



---

COMPARE


In [124]:
class ModelEvaluator:
    def __init__(self, df, protected_attributes, label_name, privileged, unprivileged, fav, unfav, groups, num_iterations=10, oversampling_methods=None):
        self.df = df
        self.protected_attributes = protected_attributes
        self.label_name = label_name
        self.privileged = privileged
        self.unprivileged = unprivileged
        self.fav = fav
        self.unfav = unfav
        self.groups = groups
        self.num_iterations = num_iterations
        self.oversampling_methods = oversampling_methods if oversampling_methods is not None else ['none']

    def evaluate_model_performance_mean(self):
        results_dict = {method: [] for method in self.oversampling_methods}

        for _ in range(self.num_iterations):
            data_prep = DataPreparation(self.df, self.protected_attributes, self.label_name,
                                        self.privileged, self.unprivileged, self.fav, self.unfav)
            data_prep.prepare()
            data_prep.df = data_prep.df.reset_index(drop=True)
            X_train, X_test = data_prep.X_train, data_prep.X_test
            X_train = X_train.reset_index(drop=True)
            cat_features = data_prep.cat_features
            numerical_features = data_prep.numerical_features
            reverse_group_mapping = data_prep.create_group_column()
            group_counts_train = X_train['Group'].value_counts().sort_index()
            subgroup_column_train = X_train['Group']
            subgroup_column_test = X_test['Group']
            X_train = X_train.drop(columns=['Group'])
            X_test = X_test.drop(columns=['Group'])

            num_privileged_ones = X_train[(X_train[self.protected_attributes[0]] == 1) &
                                          (X_train[self.protected_attributes[1]] == 1) &
                                          (X_train[self.label_name] == 1)].shape[0]
            num_privileged_zeros = X_train[(X_train[self.protected_attributes[0]] == 1) &
                                          (X_train[self.protected_attributes[1]] == 1) &
                                          (X_train[self.label_name] == 0)].shape[0]
            total_ratio = num_privileged_ones / num_privileged_zeros if num_privileged_zeros != 0 else float('inf')

            for method in self.oversampling_methods:
                X_train_method = X_train.copy()
                if method == 'none':
                    results, pred_labels = evaluate_model_performance(X_train_method, X_test, self.protected_attributes, self.label_name,
                                                                      self.groups, model=model)
                elif method == 'custom_smote_km':
                    X_train_no_sens = X_train_method.drop(columns=[self.protected_attributes[0], self.protected_attributes[1], self.label_name])
                    X_reduced = X_train_no_sens

                    kmeans = KMeans(n_clusters=5, init='k-means++', n_init=10, max_iter=300, random_state=42)
                    clusters = kmeans.fit_predict(X_reduced)
                    X_train_method['Cluster_Labels'] = clusters

                    all_synthetic_samples_km = oversample_clusters(X_train_method, 'Cluster_Labels', self.protected_attributes,
                                                                   self.label_name, total_ratio, cat_features)
                    X_train_method = X_train_method.drop(columns=['Cluster_Labels'])
                    X_train_resampled_km = pd.concat([X_train_method, all_synthetic_samples_km], ignore_index=True)
                    results, pred_labels = evaluate_model_performance(X_train_resampled_km, X_test, self.protected_attributes,
                                                                      self.label_name, self.groups, model=model)
                elif method == 'custom_smote_dbscan':
                    synthetic_samples_matrix_dbscan, synthetic_samples_group_dbscan = oversample_groups(X_train_method, cat_features, custom_smote_dbscan, subgroup_column_train, total_ratio, reverse_group_mapping)
                    X_train_resampled_dbscan = pd.concat([X_train_method, pd.DataFrame(synthetic_samples_matrix_dbscan, columns=X_train.columns)], ignore_index=True)
                    results, pred_labels = evaluate_model_performance(X_train_resampled_dbscan, X_test, self.protected_attributes,
                                                                      self.label_name, self.groups, model=model)
                elif method == 'custom_smote_tax':
                    synthetic_samples_matrix_tax, synthetic_samples_group_tax = oversample_groups(X_train_method, cat_features, custom_smote_tax, subgroup_column_train, total_ratio, reverse_group_mapping)
                    X_train_resampled_tax = pd.concat([X_train_method, pd.DataFrame(synthetic_samples_matrix_tax, columns=X_train.columns)], ignore_index=True)
                    results, pred_labels = evaluate_model_performance(X_train_resampled_tax, X_test, self.protected_attributes,
                                                                      self.label_name, self.groups, model=model)
                elif method == 'Fair-SMOTE':
                    X_train_resampled_fair_smote = oversample_fair_smote(X_train_method, self.protected_attributes, self.label_name)
                    results, pred_labels = evaluate_model_performance(X_train_resampled_fair_smote, X_test, self.protected_attributes,
                                                                      self.label_name, self.groups, model=model)
                elif method == 'Reweighing':
                    custom_data = CustomDataset(X_train_method, self.protected_attributes, self.label_name)
                    privileged_groups = [{self.protected_attributes[0]: 1, self.protected_attributes[1]: 1}]
                    unprivileged_groups = [{self.protected_attributes[0]: 0, self.protected_attributes[1]: 0}]
                    RW = Reweighing(unprivileged_groups=unprivileged_groups,
                                    privileged_groups=privileged_groups)
                    RW.fit(custom_data)
                    dataset_transf_train = RW.transform(custom_data)

                    results, pred_labels = evaluate_model_performance(X_train_method, X_test, self.protected_attributes, self.label_name,
                                                                      self.groups, model=model, weights=custom_data.instance_weights)
                elif method == 'Remedy':
                    columns_all = X_train_method.drop(columns=[self.label_name])
                    label_y = self.label_name
                    columns_protected = self.protected_attributes
                    temp2, names = get_temp(X_train_method, columns_protected, label_y)
                    unfair_group, unfair_names, skew_candidates, unfair_dict = get_unfair_group(columns_protected, [])
                    print(unfair_group, unfair_names, skew_candidates, unfair_dict)
                    all_names = candidate_groups(skew_candidates, unfair_dict, columns_protected, unfair_names)
                    names_values = name_val_dict(X_train_method, names)

                    all_names_lst = list(all_names.keys())[1:]
                    all_names_lst.reverse()
                    filter_count = 30
                    new_train_data = copy.deepcopy(X_train_method)

                    for a in all_names_lst:
                        temp2, names = get_temp(new_train_data, all_names[a], label_y)
                        temp, temp_g = get_temp_g(new_train_data, names, label_y)
                        temp_g = temp_g[temp_g['cnt'] > filter_count]
                        lst_of_counts = compute_lst_of_counts(temp, names, label_y)
                        need_pos, need_neg = compute_problematic_opt(temp2, temp_g, names, label_y, lst_of_counts)
                        new_train_data['skewed'] = 0
                        new_train_data["diff"] = 0
                        new_train_data = naive_duplicate(new_train_data, temp2, names, need_pos, need_neg, label_y)
                    new_train_label = pd.DataFrame(new_train_data, columns=[label_y])
                    new_train_label = new_train_label[label_y]
                    new_train_label = new_train_label.astype('int')

                    X_train_resampled_remedy = new_train_data.drop(columns=['skewed', 'diff'])

                    results, pred_labels = evaluate_model_performance(X_train_resampled_remedy, X_test, self.protected_attributes,
                                                                      self.label_name, self.groups, model=model)

                data_frames = []
                for key, values in results.items():
                    df_part = pd.DataFrame([values], index=[key])
                    data_frames.append(df_part)

                results_df = pd.concat(data_frames)
                results_df.index.name = 'Comparison'
                results_dict[method].append(results_df)

        combined_results = {method: pd.concat(results_dict[method]).groupby(level=0).mean() for method in self.oversampling_methods}
        return combined_results


In [ ]:
# evaluators = {
#     'ModelEvaluator': ModelEvaluator(df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label, groups, num_iterations=10, oversampling_methods=['none', 'custom_smote_dbscan', 'Fair-SMOTE', 'Reweighing', 'Remedy'])
# }

evaluators = {
    'ModelEvaluator': ModelEvaluator(df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label, groups, num_iterations=10, oversampling_methods=['none', 'custom_smote_dbscan'])
}

results = {}

# Evaluate all models and store results
for key, evaluator in evaluators.items():
    results[key] = evaluator.evaluate_model_performance_mean()

# Display the results
for method, result in results['ModelEvaluator'].items():
    print(f"Results for {method} method:")
    print(result)

Detected 373090 rows with missing values. Removed them.
The 'disposition' column has only two unique values.


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/1279187019.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df.loc[:, self.label] = self.df[self.label].replace([self.unfav, self.fav], [0, 1])
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/1279187019.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df.loc[:, self.sensitive[0]] = self.df[self.sensitive[0]].replace(race_mapping)


{'gender': {0: 0, 1: 1}, 'race': {0: 0, 1: 1}, 'maritalstatus': {'Civil Union': 0, 'Divorced': 1, 'Legally Separated': 2, 'Life Partner': 3, 'Married': 4, 'Other': 5, 'Significant Other': 6, 'Single': 7, 'Widowed': 8}, 'employstatus': {'Disabled': 0, 'Full Time': 1, 'Not Employed': 2, 'On Active Military Duty': 3, 'Part Time': 4, 'Retired': 5, 'Self Employed': 6, 'Student - Full Time': 7, 'Student - Part Time': 8}, 'insurance_status': {'Commercial': 0, 'Medicaid': 1, 'Medicare': 2, 'Other': 3, 'Self pay': 4}, 'disposition': {0: 0, 1: 1}, 'arrivalmode': {'Car': 0, 'Other': 1, 'Police': 2, 'Public Transportation': 3, 'Walk-in': 4, 'Wheelchair': 5, 'ambulance': 6}, 'previousdispo': {'AMA': 0, 'Admit': 1, 'Discharge': 2, 'Eloped': 3, 'LWBS after Triage': 4, 'LWBS before Triage': 5, 'No previous dispo': 6, 'Observation': 7, 'Send to L&D': 8, 'Transfer to Another Facility': 9}, 'top_meds': {'meds_analgesicandantihistaminecombination': 0, 'meds_analgesics': 1, 'meds_anesthetics': 2, 'meds_ant

/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/1279187019.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df['Group'] = pd.MultiIndex.from_frame(self.df[self.sensitive + [self.label]]).map(group_mapping)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/1279187019.py:148: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.91199722  0.43849355 -0.91199722 ...  1.78898433  0.43849355
  0.43849355]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.X_train.loc[:, self.columns_numerical] = train_dataset_scaled_numerical
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_84592/1279187019

[(0, (0, 0, 1)), (1, (0, 0, 0)), (2, (0, 1, 1)), (3, (0, 1, 0)), (4, (1, 0, 1)), (5, (1, 0, 0)), (6, (1, 1, 1)), (7, (1, 1, 0))]


...

{'name': 'Others Male', 'attributes': {'race': 0, 'gender': 'Male'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


{'name': 'White Female', 'attributes': {'race': 1, 'gender': 'Female'}} {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 'Female'}} gruppi


/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:278: RuntimeWarning: invalid value encountered in scalar divide
  TPR=TP / P, TNR=TN / N, FPR=FP / N, FNR=FN / P,
/Users/hakim/git/student_work/OSFFR/osffr_venv/lib/python3.10/site-packages/aif360/metrics/classification_metric.py:279: RuntimeWarning: invalid value encountered in scalar divide
  GTPR=GTP / P, GTNR=GTN / N, GFPR=GFP / N, GFNR=GFN / P,


0.555 1 [0] mids
0.01 0.555 min, max
0.28250000000000003 1 [0] mids
0.01 0.28250000000000003 min, max
0.14625000000000002 2 [-1  0] mids
0.01 0.14625000000000002 min, max
0.07812500000000001 8 [-1  0  1  2  3  4  5  6] mids
0.07812500000000001 0.14625000000000002 min, max
0.11218750000000002 2 [-1  0] mids
0.07812500000000001 0.11218750000000002 min, max
0.09515625000000003 4 [-1  0  1  2] mids
0.09515625000000003 0.11218750000000002 min, max
0.10367187500000002 5 [-1  0  1  2  3] mids
0.10367187500000002 0.11218750000000002 min, max
0.10792968750000002 3 [-1  0  1] mids
0.10792968750000002 0.11218750000000002 min, max
0.11005859375000002 2 [-1  0] mids
0.10792968750000002 0.11005859375000002 min, max
0.10899414062500001 2 [-1  0] mids
0.10792968750000002 0.10899414062500001 min, max
0.10846191406250003 3 [-1  0  1] mids
0.10846191406250003 0.10899414062500001 min, max
Oversampling for group pair (0, 1): Added 7719 synthetic samples in 0.


In [ ]:
# Print all results in a formatted way
for key, result in results.items():
    print(f"Results for {key}:")
    print(result)
    print("\n" + "-"*40 + "\n")  # Print a separator for better readability

In [ ]:

# comparison_key = 'Male Adult vs Female Young'
comparison_key = 'Caucasian Female vs Black Male'
# comparison_key = 'White Male vs Black Female'

# Create an empty list to store the extracted data
data = []

# Iterate over the results dictionary and extract the relevant data
for method, result_dict in results['ModelEvaluator'].items():
    if comparison_key in result_dict.index:
        metrics = result_dict.loc[comparison_key]
        row = {
            'Classifier': model,
            'Technique': method,
            'DI Ratio': metrics['Disparate Impact Ratio'],
            # 'AEO Diff.': metrics['Average Odds Difference'],
            'Equal Opportunity Difference': metrics['Equal Opportunity Difference'],
            'Consis.': metrics['Consistency'],
            'Acc.': metrics['Accuracy'],
            'Bal. Acc.': metrics['Balanced Accuracy'],
            'F1 Score': metrics['F1 Score']
        }
        data.append(row)

# Convert the extracted data into a DataFrame
df_results = pd.DataFrame(data)

# Print the DataFrame as a formatted table
print(df_results.to_string(index=False))
